In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, callback_context
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module_Module5 import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Credentials are read from environment variables rather than hardcoded, so
# they never end up committed to source control alongside the notebook.
# Set MONGO_USER / MONGO_PASS in your shell before launching Jupyter, e.g.:
#   export MONGO_USER=aacuser
#   export MONGO_PASS=your-password-here
username = os.environ.get("MONGO_USER")
password = os.environ.get("MONGO_PASS")
if not username or not password:
    raise RuntimeError(
        "Set the MONGO_USER and MONGO_PASS environment variables before running "
        "this notebook (do not hardcode credentials in source control)."
    )

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Create the indexes the dashboard's filter/geolocation queries rely on.
# Safe to call every run; existing indexes are left untouched.
db.create_indexes()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date


app.layout = html.Div(style={'fontFamily': 'Arial, sans-serif', 'padding': '20px', 'backgroundColor': '#f5f5f5'}, children=[
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),style={
        'height': '150px',           # Set height
        'width': '200px',            # Set width
        'margin': '10px',            # Space around image
        'display': 'block',          # Display as block element
        'margin-left': 'auto',       # Center horizontally (with margin-right)
        'margin-right': 'auto'  }),
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard')),style={'color': '#2c3e50', 'marginBottom': '10px'}),
    html.Center(html.P('Austin Animal Center Outcomes', 
                       style={'color': '#7f8c8d', 'fontSize': '18px', 'marginTop': '0'})),
    html.Hr(style={'border': '2px solid #3498db'}),
    html.Div(className='buttonRow',
            style={'display': 'flex', 'gap': '10px', 'marginBottom': '20px', 'justifyContent': 'center'},
            children=[
            html.Button('🌊 Water', id='water', n_clicks=0,
                       style={'padding': '10px 20px', 'fontSize': '16px', 'backgroundColor': '#3498db', 
                              'color': 'white', 'border': 'none', 'borderRadius': '5px', 
                              'cursor': 'pointer', 'fontWeight': 'bold'}),
            html.Button('⛰️ Mountain or Wilderness', id='mountain', n_clicks=0,
                       style={'padding': '10px 20px', 'fontSize': '16px', 'backgroundColor': '#2ecc71', 
                              'color': 'white', 'border': 'none', 'borderRadius': '5px', 
                              'cursor': 'pointer', 'fontWeight': 'bold'}),
            html.Button('🌪️ Disaster or Individual Tracking', id='track', n_clicks=0,
                       style={'padding': '10px 20px', 'fontSize': '16px', 'backgroundColor': 'red', 
                              'color': 'white', 'border': 'none', 'borderRadius': '5px', 
                              'cursor': 'pointer', 'fontWeight': 'bold'}),
            html.Button('🔄 Reset', id='submit-button-reset', n_clicks=0,
                       style={'padding': '10px 20px', 'fontSize': '16px', 'backgroundColor': '#95a5a6', 
                              'color': 'white', 'border': 'none', 'borderRadius': '5px', 
                              'cursor': 'pointer', 'fontWeight': 'bold'})]),
    html.Div(style={'fontFamily': 'Arial, sans-serif','overflowX': 'scroll'}, className='dataTable',children=[
    dash_table.DataTable(
        
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        #FIXME: Set up the features for your interactive data table to make it user-friendly for your client
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10
    ),
    html.Br(),
    html.Hr()
    
]),
    
html.H2("Animal GPS 📍", style={'color': '#2c3e50', 'marginTop': '0'}),
dcc.Loading(id='loading-map', type='default', children=html.Div(
            id='map-id',
            className='col s12 m6',
    
            )),
html.H2("Breed Distribution 📊", style={'color': '#2c3e50', 'marginTop': '20px'}),
dcc.Loading(id='loading-graph', type='default', children=html.Div(id='graph-id')),
html.Div(style={'fontFamily': 'Arial, sans-serif','overflowX': 'scroll'}, className='footer', children=html.Center(html.H6("Built with ❤️ by Jesel Reyes")))

])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(Output('datatable-id','data'),[Input('water','n_clicks'),
    Input('mountain','n_clicks'),
    Input('track','n_clicks'),                                       
    Input('submit-button-reset','n_clicks')])
def update_dashboard(water_clicks, mountain_clicks, track_clicks,reset_clicks):
    if not callback_context.triggered:
        # No button clicked yet, return all data
        return df.to_dict('records')

    button_id = callback_context.triggered[0]['prop_id'].split('.')[0]

    # water/mountain/track are handled by a single index-backed aggregation
    # query (AnimalShelter.get_rescue_candidates) instead of three duplicated
    # pandas boolean masks scanning the full in-memory dataframe.
    if button_id in ('water', 'mountain', 'track'):
        results = db.get_rescue_candidates(button_id)
        filtered_df = pd.DataFrame.from_records(results)
        if '_id' in filtered_df.columns:
            filtered_df.drop(columns=['_id'], inplace=True)
        return filtered_df.to_dict('records')

    # Reset button (or any other trigger) - show all animals
    return df.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) 
    
    # Check if data exists
    if viewData is None or len(viewData) == 0:
        return [dcc.Graph(figure={})]
    
    # converting the dictionary into a dataframe
    dff = pd.DataFrame.from_dict(viewData)
    if 'breed' not in dff.columns:
        return [dcc.Graph(figure={})]
     #get the distinct breed names and their counts and convert back into a dataframe
    breed_counts = dff['breed'].value_counts().head(10).reset_index()
    #renaming column names in dataframe
    breed_counts.columns = ['breed','count']
    return [
       dcc.Graph(            
           figure = px.pie(breed_counts, names='breed',values='count')
       )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data available")]
# #FIXME Add in the code for your geolocation chart
    dff = pd.DataFrame.from_dict(viewData)
 # Because we only allow single row selection, the list can 
 # be converted to a row index here
    
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    if row >= len(dff):
        row = 0
        
        # Get location data with error handling
    try:
        lat = dff.iloc[row]['location_lat']  # location_lat
        lon = dff.iloc[row]['location_long']  # location_long
        breed = dff.iloc[row]['breed']  # breed
        name = dff.iloc[row]['name']   # name
        
        # Check if coordinates are valid
        if pd.isna(lat) or pd.isna(lon):
            return [html.Div("No location data available for this animal")]
        
    except (IndexError, KeyError) as e:
        return [html.Div(f"Error accessing data: {str(e)}")]
        
    return [
    dl.Map(style={'width': '100%', 'height': '500px','max-width': '1000px'},
           center=[lat,lon], zoom=15, children=[
           
           dl.TileLayer(id="base-layer-id"),
           # Marker with tool tip and popup
           # Column 13 and 14 define the grid-coordinates for 
           # the map
           # Column 4 defines the breed for the animal
           # Column 9 defines the name of the animal
           dl.Marker(position=[lat,lon],
              children=[
              dl.Tooltip(breed),
              dl.Popup([
                 html.H4("Animal Name"),
                 html.P(name)
              ])
           ])
        ])
]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='jupyterlab') 